# Evaluación Evolutiva del Sistema de Detección de Carriles

Este cuaderno documenta el proceso completo de validación y mejora de nuestro algoritmo de detección. Para demostrar el rigor científico del proyecto, no solo evaluaremos una versión final, sino que compararemos la evolución del sistema a través de tres arquitecturas distintas (Pipelines):

*   **V1 (Baseline Original):** Implementación estricta del artículo de referencia. Modo Día usando Transformada Estándar de Hough (SHT) y Modo Noche usando Transformada Probabilística (PHT).
*   **V2 (Unificación PHT):** Modificación de la arquitectura diurna para utilizar PHT, buscando mayor eficiencia computacional y consistencia entre ambos modos.
*   **V3 (Mejora Propia - Tracking TTL):** Implementación de un filtro de paso bajo (Time-To-Live Tracker) diseñado por nosotros para mitigar la ceguera temporal de los sensores mediante imputación temporal con prevención de deriva.

### Las Métricas de Validación
El rendimiento se evalúa fotograma a fotograma calculando:
1. **Detección de ambas líneas:** Ratio de éxito bruto (Frames detectados / Total frames).
2. **Pendiente Media ($Slope$):** Promedio de inclinación para validar la perspectiva geométrica.
3. **Distancia Media ($Distance$):** Separación en píxeles en la base de la imagen.
4. **Consistencia Temporal ($Consistency$):** Estabilidad del trazado limitando la variación angular entre fotogramas consecutivos a un máximo del 15%.


### 1. Preparación y Carga de Datos (Data Ingestion)

Para que nuestra evaluación sea estadísticamente significativa, necesitamos someter el algoritmo a una prueba de estrés con cientos de imágenes. En esta sección, implementamos un cargador masivo que recorrerá de forma automatizada las subcarpetas de nuestras bases de datos (TuSimple para el día y BD2 para la noche).

Para proteger la memoria RAM del equipo y asegurar tiempos de cómputo viables durante la validación, aplicamos un límite de seguridad de `max_frames` (ej. 2000 imágenes por entorno).

In [1]:
# ========================================================
# 1. CARGA MASIVA DE DATOS Y LIBRERÍAS
# ========================================================
import cv2
import numpy as np
import math
import pandas as pd
import os

# Ajustar a las rutas locales reales
RUTA_TUSIMPLE = r"C:\Users\hugot\Downloads\archive\TUSimple\test_set\clips\0531"
RUTA_NOCTURNA = r"C:\Users\hugot\Downloads\BD2\BD2"                            

def cargar_secuencia_masiva(ruta_base, max_frames=2000):
    """Extrae todos los fotogramas de las subcarpetas con un límite de seguridad."""
    fotogramas_totales = []
    carpetas_videos = [os.path.join(ruta_base, d) for d in os.listdir(ruta_base) 
                       if os.path.isdir(os.path.join(ruta_base, d))]
    
    if not carpetas_videos: 
        print(f"Error: No se encontraron carpetas en {ruta_base}.")
        return []
    
    for carpeta in carpetas_videos:
        frames_clip = sorted([os.path.join(carpeta, f) for f in os.listdir(carpeta) 
                              if f.endswith('.jpg') or f.endswith('.png')])
        fotogramas_totales.extend(frames_clip)
        
    muestra = fotogramas_totales[:max_frames]
    print(f"[OK] {os.path.basename(ruta_base)} cargado: {len(muestra)} frames para evaluar.")
    return muestra

# Ejecutamos la carga en memoria
frames_dia = cargar_secuencia_masiva(RUTA_TUSIMPLE, max_frames=2000)
frames_noche = cargar_secuencia_masiva(RUTA_NOCTURNA, max_frames=2000)

[OK] 0531 cargado: 2000 frames para evaluar.
[OK] BD2 cargado: 2000 frames para evaluar.


### 2. Motores de Extracción Matemática (Visión Pura)

En lugar de procesar matrices de imagen pesadas para dibujar líneas visuales, estos motores están diseñados para extraer la matemática cruda (pendiente y punto de corte de las rectas). Hemos modularizado tres motores distintos para nuestra comparativa:

1. **SHT Diurno (Versión 1):** Aplica la Transformada Estándar de Hough y requiere conversión de coordenadas polares a cartesianas.
2. **PHT Diurno (Versión 2 y 3):** Sustituye la SHT por la Transformada Probabilística (PHT) para ganar eficiencia computacional, calculando el promedio de los segmentos detectados.
3. **PHT Nocturno (Común):** Adapta la visión a condiciones de baja luminosidad seleccionando estadísticamente el segmento más cercano al centroide del vehículo en lugar de promediar.

In [2]:
# ========================================================
# 2. MOTORES DE EXTRACCIÓN (SHT vs PHT)
# ========================================================

def extraccion_sht_dia(img_roi):
    """Motor Original: Hough Estándar (Polar a Cartesiano)"""
    lineas = cv2.HoughLines(img_roi, rho=1, theta=np.pi/180, threshold=80)
    p_izq, c_izq, p_der, c_der = [], [], [], []
    
    if lineas is not None:
        for linea in lineas:
            rho, theta = linea[0]
            if math.sin(theta) == 0: continue
            slope = -math.cos(theta) / math.sin(theta)
            intercept = rho / math.sin(theta)
            
            if slope < -0.5: p_izq.append(slope); c_izq.append(intercept)
            elif slope > 0.5: p_der.append(slope); c_der.append(intercept)
            
    if p_izq and p_der: return np.mean(p_izq), np.mean(c_izq), np.mean(p_der), np.mean(c_der)
    return None, None, None, None

def extraccion_pht_dia(img_roi):
    """Motor V2: Hough Probabilístico (Cartesiano Directo + Averaging)"""
    lineas = cv2.HoughLinesP(img_roi, 1, np.pi/180, 20, minLineLength=20, maxLineGap=300)
    p_izq, c_izq, p_der, c_der = [], [], [], []
    
    if lineas is not None:
        for linea in lineas:
            x1, y1, x2, y2 = linea[0]
            if x1 == x2: continue 
            slope = (y2 - y1) / (x2 - x1)
            intercept = y1 - slope * x1
            if slope < -0.5: p_izq.append(slope); c_izq.append(intercept)
            elif slope > 0.5: p_der.append(slope); c_der.append(intercept)

    if p_izq and p_der: return np.mean(p_izq), np.mean(c_izq), np.mean(p_der), np.mean(c_der)
    return None, None, None, None

def extraccion_pht_noche(img_roi, centro_x):
    """Motor Noche (Compartido en las 3 versiones): PHT + Closest Distance"""
    lineas = cv2.HoughLinesP(img_roi, 1, np.pi/180, 20, minLineLength=15, maxLineGap=200)
    cand_izq, cand_der = [], []
    
    if lineas is not None:
        for linea in lineas:
            x1, y1, x2, y2 = linea[0]
            if x1 == x2: continue
            m = (y2 - y1) / (x2 - x1)
            c = y1 - m * x1
            theta = abs(math.atan(m))
            d = abs(((x1 + x2) / 2) - centro_x) 
            if math.radians(20) < theta < math.radians(80):
                if m < 0: cand_izq.append({'d': d, 'm': m, 'c': c})
                else: cand_der.append({'d': d, 'm': m, 'c': c})

    if cand_izq and cand_der:
        mejor_izq = min(cand_izq, key=lambda x: x['d'])
        mejor_der = min(cand_der, key=lambda x: x['d'])
        return mejor_izq['m'], mejor_izq['c'], mejor_der['m'], mejor_der['c']
    return None, None, None, None

### 3. Preprocesado y Filtro de Seguimiento Temporal (Tracker V3)

Esta sección contiene el núcleo de nuestra innovación principal. La cámara, por sí sola, sufre de ceguera temporal ante reflejos o zonas de sombra (parpadeo o *flickering*). Para solucionarlo, implementamos un sistema de **Imputación Temporal con Prevención de Deriva**:

* **Mecanismo Time-To-Live (TTL):** El sistema dota a las líneas de una "memoria a corto plazo" de 5 fotogramas. Si el algoritmo de visión pura pierde el carril, el Tracker interviene e imputa la última trayectoria conocida.
* **Seguridad:** Si la ceguera supera los 5 fotogramas (aprox. 160ms), el sistema borra la memoria para evitar derivas peligrosas en curvas pronunciadas.

La función `procesar_frame` actuará como un enrutador inteligente, aplicando los filtros de color y enviando el ROI al motor adecuado según la versión del algoritmo que estemos evaluando.

In [3]:
# ========================================================
# 3. ENRUTADOR DE IMAGEN, PREPROCESADO Y TRACKER V3
# ========================================================

def postprocesar_tracker(m_izq_raw, c_izq_raw, m_der_raw, c_der_raw, tracker):
    """Módulo V3: Imputación Temporal con Prevención de Deriva (5 frames TTL)"""
    MAX_VIDAS = 5 
    
    # Carril Izquierdo
    if m_izq_raw is not None:
        tracker['m_izq'], tracker['c_izq'] = m_izq_raw, c_izq_raw
        tracker['ttl_izq'] = MAX_VIDAS
        m_izq_final, c_izq_final = m_izq_raw, c_izq_raw
    else:
        if tracker['ttl_izq'] > 0:
            m_izq_final, c_izq_final = tracker['m_izq'], tracker['c_izq']
            tracker['ttl_izq'] -= 1  
        else:
            m_izq_final, c_izq_final = None, None

    # Carril Derecho
    if m_der_raw is not None:
        tracker['m_der'], tracker['c_der'] = m_der_raw, c_der_raw
        tracker['ttl_der'] = MAX_VIDAS
        m_der_final, c_der_final = m_der_raw, c_der_raw
    else:
        if tracker['ttl_der'] > 0:
            m_der_final, c_der_final = tracker['m_der'], tracker['c_der']
            tracker['ttl_der'] -= 1
        else:
            m_der_final, c_der_final = None, None

    return m_izq_final, c_izq_final, m_der_final, c_der_final, tracker

def procesar_frame(img, modo_iluminacion, version_pipeline):
    """
    Se encarga de preprocesar (Filtros, Gamma, ROI) según sea Día o Noche,
    y luego desvía el ROI al motor matemático que corresponda según la Versión.
    """
    alto, ancho = img.shape[:2]
    
    if modo_iluminacion == "DIA":
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mask_amarillo = cv2.inRange(hsv, np.array([15, 80, 80]), np.array([35, 255, 255]))
        mask_blanco = cv2.inRange(hsv, np.array([0, 0, 200]), np.array([180, 30, 255]))
        img_filtrada = cv2.bitwise_and(img, img, mask=cv2.bitwise_or(mask_amarillo, mask_blanco))
        img_blur = cv2.GaussianBlur(cv2.cvtColor(img_filtrada, cv2.COLOR_BGR2GRAY), (5, 5), 0)
        img_canny = cv2.Canny(img_blur, 50, 150)
        
        poligono = np.array([[(0, alto), (ancho, alto), (int(ancho*0.75), int(alto*0.6)), (int(ancho*0.25), int(alto*0.6))]])
        mask_roi = np.zeros_like(img_canny)
        cv2.fillPoly(mask_roi, poligono, 255)
        img_roi = cv2.bitwise_and(img_canny, mask_roi)
        
        # Enrutador Versión Día
        if version_pipeline == "V1_ORIGINAL":
            return extraccion_sht_dia(img_roi)
        else: # V2 y V3 usan PHT en Día
            return extraccion_pht_dia(img_roi)
            
    else: # MODO NOCHE
        gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        y_average = np.mean(gris)
        gamma_final = 0.7 - (-0.3 / math.log10(max(y_average, 1e-6)))
        tabla = np.array([((i / 255.0) ** (1.0/gamma_final)) * 255 for i in np.arange(0, 256)]).astype("uint8")
        img_gamma = cv2.LUT(img, tabla)
        
        hsv = cv2.cvtColor(img_gamma, cv2.COLOR_BGR2HSV)
        mask_blanco = cv2.inRange(hsv, np.array([0, 0, 180]), np.array([180, 30, 255]))
        mask_amarillo = cv2.inRange(hsv, np.array([15, 80, 80]), np.array([35, 255, 255]))
        mask_total = cv2.bitwise_or(mask_blanco, mask_amarillo)
        
        poligono = np.array([[(0, alto), (ancho, alto), (int(ancho*0.75), int(alto*0.7)), (int(ancho*0.25), int(alto*0.7))]])
        mask_roi = np.zeros_like(mask_total)
        cv2.fillPoly(mask_roi, poligono, 255)
        img_roi = cv2.bitwise_and(mask_total, mask_roi)
        
        # Enrutador Versión Noche (Todas usan PHT)
        return extraccion_pht_noche(img_roi, centro_x=ancho/2)

### 4. Motor Universal de Evaluación de Rendimiento

Este bloque implementa estrictamente las matemáticas de evaluación propuestas para el proyecto. En lugar de repetir código, hemos diseñado una función universal que evalúa cualquier secuencia y versión inyectando la lógica correspondiente. 

Las métricas calculadas fotograma a fotograma son:
* **Detection:** Ratio de éxito en la localización simultánea de los límites izquierdo y derecho del carril.
* **Slope:** Validación de la perspectiva geométrica mediante el promedio angular absoluto.
* **Distance:** Cálculo de la distancia euclidiana en píxeles simulando la base del vehículo.
* **Consistency:** Medida de estabilidad direccional, validando que la oscilación de la pendiente sea inferior al 15% respecto al fotograma inmediatamente anterior.

In [4]:
# ========================================================
# 4. MOTOR UNIVERSAL DE EVALUACIÓN MATEMÁTICA
# ========================================================

def evaluar_secuencia_completa(rutas_frames, modo_iluminacion, version_pipeline):
    """
    Procesa toda la secuencia y calcula las 4 métricas de rendimiento.
    Comporta la lógica de activar/desactivar el Tracker según la versión.
    """
    total_frames = len(rutas_frames)
    if total_frames == 0: return {}
    
    frames_detectados = 0
    frames_consistentes = 0
    lista_distancias = []
    lista_pendientes = []
    
    # Memoria
    estado_tracker = {'m_izq': None, 'c_izq': None, 'ttl_izq': 0, 'm_der': None, 'c_der': None, 'ttl_der': 0}
    m_izq_ant, m_der_ant = None, None 
    
    for i, ruta in enumerate(rutas_frames):
        img = cv2.imread(ruta)
        if img is None: continue
        alto = img.shape[0]
        
        # 1. Extracción
        m_izq, c_izq, m_der, c_der = procesar_frame(img, modo_iluminacion, version_pipeline)
            
        # 2. Aplicar V3 Tracker (Solo si estamos en la versión 3)
        if version_pipeline == "V3_TRACKER":
            m_izq, c_izq, m_der, c_der, estado_tracker = postprocesar_tracker(
                m_izq, c_izq, m_der, c_der, estado_tracker
            )
            
        # 3. Métricas
        if m_izq is not None and m_der is not None:
            frames_detectados += 1
            
            slope_medio = (abs(m_izq) + abs(m_der)) / 2.0
            lista_pendientes.append(slope_medio)
            
            x_base_izq = (alto - c_izq) / m_izq
            x_base_der = (alto - c_der) / m_der
            lista_distancias.append(abs(x_base_der - x_base_izq))
            
            if m_izq_ant is not None and m_der_ant is not None:
                div_izq = abs(m_izq_ant) if m_izq_ant != 0 else 1e-6
                div_der = abs(m_der_ant) if m_der_ant != 0 else 1e-6
                if (abs((m_izq - m_izq_ant) / div_izq) < 0.15) and (abs((m_der - m_der_ant) / div_der) < 0.15):
                    frames_consistentes += 1
            
            m_izq_ant, m_der_ant = m_izq, m_der
        else:
            m_izq_ant, m_der_ant = None, None

    return {
        "Consistency": frames_consistentes / total_frames,
        "Distance (px)": np.mean(lista_distancias) if lista_distancias else 0,
        "Slope": np.mean(lista_pendientes) if lista_pendientes else 0,
        "Detection": frames_detectados / total_frames
    }

### 5. Auditoría Final: Ejecución y Resultados Evolutivos

Procedemos a la ejecución integral del banco de pruebas. Someteremos las muestras de imágenes diurnas y nocturnas a las tres versiones evolutivas de nuestro sistema:

* **V1 (Original):** Replicación de la arquitectura base SHT/PHT.
* **V2 (Optimización PHT):** Unificación de algoritmos hacia la Transformada Probabilística.
* **V3 (State-of-the-art):** Sistema final integrando el *Tracker* temporal.

Los resultados se compilan mediante la librería Pandas en una única tabla de rendimiento cruzado, permitiendo observar empíricamente el impacto de nuestras decisiones de ingeniería en la robustez del sistema autónomo.

In [5]:
# ========================================================
# 5. EJECUCIÓN MULTI-PIPELINE Y GENERACIÓN DE TABLA FINAL
# ========================================================
from IPython.display import display

print("Iniciando auditoría completa del algoritmo...")

# 1. Evaluar las 3 Versiones en MODO DÍA
print("\nEvaluando Modo DÍA (TuSimple)...")
res_dia_v1 = evaluar_secuencia_completa(frames_dia, "DIA", "V1_ORIGINAL")
res_dia_v2 = evaluar_secuencia_completa(frames_dia, "DIA", "V2_PHT_ONLY")
res_dia_v3 = evaluar_secuencia_completa(frames_dia, "DIA", "V3_TRACKER")

# 2. Evaluar las 3 Versiones en MODO NOCHE
print("Evaluando Modo NOCHE (BD2)...")
res_noche_v1 = evaluar_secuencia_completa(frames_noche, "NOCHE", "V1_ORIGINAL")
res_noche_v2 = evaluar_secuencia_completa(frames_noche, "NOCHE", "V2_PHT_ONLY") # Igual a V1 por defecto
res_noche_v3 = evaluar_secuencia_completa(frames_noche, "NOCHE", "V3_TRACKER")

print("\n--- CÁLCULOS FINALIZADOS ---")

# 3. Construcción de la Tabla Definitiva con Pandas
df_comparativa = pd.DataFrame({
    "Metric": ["Detection Rate", "Consistency", "Slope (Mean)", "Distance (px)"],
    
    "Día (V1: SHT)": [
        f"{res_dia_v1['Detection']:.2f}", f"{res_dia_v1['Consistency']:.2f}", 
        f"{res_dia_v1['Slope']:.2f}", f"{res_dia_v1['Distance (px)']:.1f}"
    ],
    "Día (V2: PHT)": [
        f"{res_dia_v2['Detection']:.2f}", f"{res_dia_v2['Consistency']:.2f}", 
        f"{res_dia_v2['Slope']:.2f}", f"{res_dia_v2['Distance (px)']:.1f}"
    ],
    "Día (V3: PHT + Tracker)": [
        f"{res_dia_v3['Detection']:.2f}", f"{res_dia_v3['Consistency']:.2f}", 
        f"{res_dia_v3['Slope']:.2f}", f"{res_dia_v3['Distance (px)']:.1f}"
    ],
    
    "Noche (V1/V2: PHT)": [
        f"{res_noche_v1['Detection']:.2f}", f"{res_noche_v1['Consistency']:.2f}", 
        f"{res_noche_v1['Slope']:.2f}", f"{res_noche_v1['Distance (px)']:.1f}"
    ],
    "Noche (V3: PHT + Tracker)": [
        f"{res_noche_v3['Detection']:.2f}", f"{res_noche_v3['Consistency']:.2f}", 
        f"{res_noche_v3['Slope']:.2f}", f"{res_noche_v3['Distance (px)']:.1f}"
    ]
})

print("\n========================================================================")
print(" TABLE 4: EVOLUTIONARY PERFORMANCE METRICS ACROSS PIPELINES (N=2000)")
print("========================================================================")
display(df_comparativa.style.set_properties(**{'text-align': 'center'}))

Iniciando auditoría completa del algoritmo...

Evaluando Modo DÍA (TuSimple)...
Evaluando Modo NOCHE (BD2)...

--- CÁLCULOS FINALIZADOS ---

 TABLE 4: EVOLUTIONARY PERFORMANCE METRICS ACROSS PIPELINES (N=2000)


,Metric,Día (V1: SHT),Día (V2: PHT),Día (V3: PHT + Tracker),Noche (V1/V2: PHT),Noche (V3: PHT + Tracker)
0,Detection Rate,0.28,0.58,0.76,0.65,0.82
1,Consistency,0.21,0.45,0.66,0.51,0.68
2,Slope (Mean),1.03,1.19,1.39,0.77,0.77
3,Distance (px),1062.5,1039.9,1028.3,1009.6,1010.5
